# Quickstart: one company, one filing

Find Apple's latest annual report on SEC EDGAR, see what it contains, read it,
and pull its tagged facts. Every object keeps the source URL it came from.

## Identify your SEC requests

The SEC asks automated clients for a name or organization and a contact email,
separated by a space. For example, `Alex Morgan alex@example.com` or
`Example Research filings@example.com`. These are examples: use your own name
and a working email. No SEC account or API key is needed.

In the next cell, uncomment the `os.environ[...]` line and replace the example
with your details. If you already set this environment variable in your
terminal, leave the line commented. The label is sent to the SEC with each
request. See the [setup guide](../../docs/getting-started.md#sec-request-identity).

In [1]:
import os
from collections import Counter

from opencorpfilings import Company, provider_info, search_companies

# Uncomment this line and replace the example with your own name and contact email.
# os.environ["OPENCORPFILINGS_SEC_USER_AGENT"] = "Alex Morgan alex@example.com"
assert os.getenv("OPENCORPFILINGS_SEC_USER_AGENT"), "Uncomment the SEC identity example above and replace it with your own details, or set OPENCORPFILINGS_SEC_USER_AGENT in your environment."

Each market route declares what it accepts. `ocf markets` prints the same for every route.

In [2]:
route = provider_info("US")
route.tier, route.identifiers, route.forms[:3]

('supported', ('ticker', 'CIK'), ('annual', 'quarterly', 'interim'))

A category such as `annual` returns the original report, not a later amendment.

In [3]:
apple = Company("AAPL", market="US")
filing = apple.filings(form="annual").latest()
filing

type,10-K
category,ANNUAL
filing_date,2025-10-31
published_at,2025-10-31 14:01:26+00:00
period,to 2025-09-27
amendment,False
market,US
regulator,US Securities and Exchange Commission
provider,sec
source,https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/0000320193-25-000079-index.htm


In [4]:
filing.to_dict()["filing_date"], filing.period_end, filing.source_url

('2025-10-31', datetime.date(2025, 9, 27), 'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/0000320193-25-000079-index.htm')

An EDGAR filing lists many files. Each has a role, and `primary_document` is the
report itself, not one of the files EDGAR generates.

In [5]:
Counter(document.role for document in filing.documents)

Counter({'generated': 78, 'exhibit': 6, 'data': 6, 'other': 2, 'report': 1})

In [6]:
filing.primary_document

document_id,0000320193-25-000079:aapl-20250927.htm
filing_id,0000320193-25-000079
document_type,10-K
role,report
media_type,text/html
description,10-K
primary,True
source,https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm


Text of the report, and one Item located in the original:

In [7]:
text = filing.text()
print(len(text), "characters")
print(text[:300])

208385 characters
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EX


In [8]:
item_7 = filing.section("Item 7")
print(item_7.title)
print(item_7.text[:400])

Management’s Discussion and Analysis of Financial Condition and Results of Operations
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations
The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, Item 8 of this Form 10-K. This Item generally discusses 2025 and 2024 items and year-to-year comparisons between 2025 and 2024. Discussions of 2023 items and year-t


Tagged XBRL facts, values exactly as filed. `facts_to_pandas()` needs the `pandas` extra.

In [9]:
facts = filing.facts_to_pandas()
facts[facts["concept_raw"] == "us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax"][
    ["value", "unit", "period_start", "period_end"]].head()

,value,unit,period_start,period_end
89,307003000000,iso4217:USD,2024-09-29,2025-09-27
90,294866000000,iso4217:USD,2023-10-01,2024-09-28
91,298085000000,iso4217:USD,2022-09-25,2023-09-30
92,109158000000,iso4217:USD,2024-09-29,2025-09-27
93,96169000000,iso4217:USD,2023-10-01,2024-09-28
